## Importing the Necessary Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
from scipy.spatial.distance import cdist
warnings.filterwarnings('ignore')

# Using seaborn set_style function for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## Loading the Data

In [2]:
data = pd.read_csv('kmeans_data/data.csv', header=None)
labels = pd.read_csv('kmeans_data/label.csv', header=None)

In [3]:
data.head()

,0,1,2,3,4,5,6,7,8,9,...,774,775,776,777,778,779,780,781,782,783
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
labels

,0
0,7
1,2
2,1
3,0
4,4
...,...
9995,2
9996,3
9997,4
9998,5


In [5]:
print(data.shape)
print(labels.shape)

(10000, 784)
(10000, 1)


## Converting the Data and Labels to NumPy Arrays

In [6]:
X = data.values
y = labels.values.flatten()

In [7]:
print(X.dtype)
print(y.dtype)

int64
int64


## Data Exploration

In [8]:
print("Number of samples: ", X.shape[0])
print("Number of features: ", X.shape[1])


Number of samples:  10000
Number of features:  784


In [9]:
print("Unique Classes in Labels: ", np.unique(y))
print("Number of Unique Classes: ", len(np.unique(y)))

Unique Classes in Labels:  [0 1 2 3 4 5 6 7 8 9]
Number of Unique Classes:  10


In [10]:
unique_classes, counts = np.unique(y, return_counts=True)

In [11]:
for label, count in zip(unique_classes, counts):
    perecentage = (count / len(y)) * 100 
    print(f"Class {label}: {count} samples, {perecentage:.2f}% of total")

K = len(unique_classes)
print(f"Total number of clusters (K): {K}")


Class 0: 980 samples, 9.80% of total
Class 1: 1135 samples, 11.35% of total
Class 2: 1032 samples, 10.32% of total
Class 3: 1010 samples, 10.10% of total
Class 4: 982 samples, 9.82% of total
Class 5: 892 samples, 8.92% of total
Class 6: 958 samples, 9.58% of total
Class 7: 1028 samples, 10.28% of total
Class 8: 974 samples, 9.74% of total
Class 9: 1009 samples, 10.09% of total
Total number of clusters (K): 10


### Question 1

## Building the K-Means Algorithm from scratch

We are required to build the K-Means algorithm using three different distance metrices: Euclidean, Cosine Similarity and Generalized Jaccard similarity. 

In [12]:
# First we will implement the Euclidean distance function

def euclidean_distance(point1, point2):
    """Calculate the Euclidean distance between two points."""
    return np.sqrt(np.sum((point1 - point2) ** 2))

In [13]:
# Now we will build the K-Means algorithm using Cosine Similarity metric

def cosine_distance(point1, point2):
    """Calculate the Cosine Similarity between two points."""

    # Calculating the dot product between the two points.
    dot_product = np.dot(point1, point2)

    norm1 = np.linalg.norm(point1)
    norm2 = np.linalg.norm(point2)

    # Avoid division by zero
    if norm1 == 0 or norm2 == 0:
        return 1.0
    
    cosine_similarity = dot_product / (norm1 * norm2)

    return 1 - cosine_similarity


In [14]:
# Implementing Generalized Jaccard Similarity

def jaccard_distance(point1, point2):
    """ Calculate the Generalized Jaccard Similarity between two points."""
    numerator = np.sum(np.minimum(point1, point2))
    denominator = np.sum(np.maximum(point1, point2))

    # Avoid division by zero
    if denominator == 0:
        return 1.0
    
    jaccard_similarity = numerator / denominator
    return 1 - jaccard_similarity


## Randomly Initializing Centroids for K-Means Algorithm

In [15]:
def initialize_centroids(X, k, random_seed=42):
    """Initialize K centroids using k-means++ strategy (Euclidean-based)."""
    np.random.seed(random_seed)

    n_samples = X.shape[0]

    # 1) Pick the first centroid completely at random
    first_idx = np.random.randint(0, n_samples)
    centroids = [X[first_idx]]

    # 2) Pick each subsequent centroid with probability
    #    proportional to the squared distance to the nearest existing centroid
    for _ in range(1, k):
        # Distances from every point to the nearest already chosen centroid
        distances = np.min(
            np.linalg.norm(X[:, np.newaxis] - np.array(centroids), axis=2) ** 2,
            axis=1
        )

        total = distances.sum()
        if total == 0:
            # All points are identical – just pick a random one
            next_idx = np.random.randint(0, n_samples)
        else:
            probabilities = distances / total
            cumulative_prob = np.cumsum(probabilities)
            r = np.random.rand()
            next_idx = np.searchsorted(cumulative_prob, r)

        centroids.append(X[next_idx])

    return np.array(centroids)


In [16]:
# Assigning the Points to Nearest Centroid

def assign_clusters(X, centroids, distance_function):
    """ Assign the data points to the nearest centroids based on the given distance function. """

    n_samples = X.shape[0]
    labels = np.zeros(n_samples, dtype=int)

    for i in range(n_samples):
        # Calculate the distance from the point to each of the centroids
        distances = [distance_function(X[i], centroid) for centroid in centroids]

        # Assigning the points to the nearest centroids
        labels[i] = np.argmin(distances)
    
    return labels


## Updating the Centroids based on the Mean

In [17]:
def update_centroids(X, labels, k):
    """Update the centroids for each of the clusters."""
    n_features = X.shape[1]
    new_centroids = np.zeros((k, n_features))

    for i in range(k):
        # Points assigned to cluster i
        cluster_points = X[labels == i]

        if len(cluster_points) > 0:
            # Use the mean of points in this cluster
            new_centroids[i] = cluster_points.mean(axis=0)
        else:
            # Reinitialize to a random data point to avoid NaN centroids
            rand_idx = np.random.randint(0, X.shape[0])
            new_centroids[i] = X[rand_idx]

    return new_centroids


In [18]:
def update_centroids_jaccard(X, labels, k):
    """
    Update centroids for Jaccard distance using medoid-style update.
    """
    n_features = X.shape[1]
    new_centroids = np.zeros((k, n_features))

    for i in range(k):
        cluster_points = X[labels == i]

        if len(cluster_points) == 0:
            # Empty cluster: reinitialize to a random data point
            rand_idx = np.random.randint(0, X.shape[0])
            new_centroids[i] = X[rand_idx]
        else:
            # Pairwise Jaccard distances within the cluster (vectorized)
            # cdist returns an (n_points x n_points) matrix
            dist_matrix = cdist(cluster_points, cluster_points, metric="jaccard")

            # Sum distance from each point to all others
            dist_sums = dist_matrix.sum(axis=1)

            # Choose the point with minimal total distance (medoid)
            best_idx = np.argmin(dist_sums)
            new_centroids[i] = cluster_points[best_idx]

    return new_centroids


## Calculating the Sum of Squared Errors for all the Distance Metrices

In [19]:
def calculate_sse(X, labels, centroids, distance_function):
    """
    Calculate the SSE.
    """
    sse = 0.0

    for i in range(len(centroids)):
        # Points assigned to cluster i
        cluster_points = X[labels == i]

        for point in cluster_points:
            d = distance_function(point, centroids[i])

            if distance_function == euclidean_distance:
                # Classic SSE
                sse += d ** 2
            else:
                # Cosine / Jaccard: sum of distances (no squaring)
                sse += d

    return sse


In [20]:
# Define the k for K-Means Algorithm
K = len(np.unique(y))

## Compiling the K-Means Algorithm

In [21]:
# Normalizing the Data
X_normalized = X / 255.0

In [22]:
def kmeans(X, k, distance_function, max_iterations=50, random_seed=42):
    """K-Means Clustering Algorithm Implementation."""
    centroids = initialize_centroids(X, k, random_seed)
    sse_history = []
    prev_sse = float("inf")

    for iteration in range(max_iterations):
        # 1. Assign clusters based on the nearest centroids
        labels = assign_clusters(X, centroids, distance_function)

        # 2. Compute current SSE / total distance
        current_sse = calculate_sse(X, labels, centroids, distance_function)
        sse_history.append(current_sse)

        # 3. Stop if SSE increased
        if current_sse > prev_sse:
            print(f"Stopped as there was increase in SSE at iteration {iteration}")
            break

        # 4. Update centroids depending on distance function
        if distance_function == jaccard_distance:
            # Jaccard: use medoid-style update
            new_centroids = update_centroids_jaccard(X, labels, k)
        else:
            # Euclidean / Cosine: mean update
            new_centroids = update_centroids(X, labels, k)

            # For cosine, normalize centroids to unit length
            if distance_function == cosine_distance:
                for j in range(k):
                    norm = np.linalg.norm(new_centroids[j])
                    if norm > 0:
                        new_centroids[j] = new_centroids[j] / norm

        # 5. Check centroid convergence
        if np.allclose(centroids, new_centroids):
            print(f"Converged at iteration {iteration}")
            iteration += 1
            break

        centroids = new_centroids
        prev_sse = current_sse
    else:
        print(f"Stopped after reaching maximum iterations: {max_iterations}")
        iteration = max_iterations

    return labels, centroids, current_sse, iteration, sse_history


## Running the K-Means Algorithm with all the 3 distance metrices

In [23]:
# 1. Euclidean Distance Metric

K = len(np.unique(y))

start_time = time.time()
euclidean_labels, euclidean_centroids, euclidean_sse, euclidean_iter, euclidean_sse_history = kmeans(X_normalized, K, euclidean_distance, max_iterations=100, random_seed=42)
euclidean_time = time.time() - start_time

print("Euclidean Distance Metric Results:")
print(f" Final SSE: {euclidean_sse}")
print(f" Number of iterations: {euclidean_iter}")
print(f" Time taken: {euclidean_time:.4f} seconds")

Converged at iteration 80
Euclidean Distance Metric Results:
 Final SSE: 391129.7599267777
 Number of iterations: 81
 Time taken: 31.9449 seconds


In [24]:
# 2. Cosine Distance Metric

start_time = time.time()
cosine_labels, cosine_centroids, cosine_sse, cosine_iter, cosine_sse_history = kmeans(X_normalized, K, cosine_distance, max_iterations=100, random_seed=42)
cosine_time = time.time() - start_time

print("Cosine Similarity Distance Metric Results:")
print(f" Final SSE: {cosine_sse}")
print(f" Number of iterations: {cosine_iter}")
print(f" Time taken: {cosine_time:.4f} seconds")

Converged at iteration 40
Cosine Similarity Distance Metric Results:
 Final SSE: 2462.8528582404065
 Number of iterations: 41
 Time taken: 19.8196 seconds


In [25]:
# 3. Generalized Jaccard Distance Metric
start_time = time.time()
jaccard_labels, jaccard_centroids, jaccard_sse, jaccard_iter, jaccard_sse_history = kmeans(X_normalized, K, jaccard_distance, max_iterations=100, random_seed=42)
jaccard_time = time.time() - start_time

print("Generalized Jaccard Distance Metric Results:")
print(f" Final SSE: {jaccard_sse}")
print(f" Number of iterations: {jaccard_iter}")
print(f" Time taken: {jaccard_time:.4f} seconds")

Converged at iteration 4
Generalized Jaccard Distance Metric Results:
 Final SSE: 5508.756354877965
 Number of iterations: 5
 Time taken: 37.3528 seconds


### Question 2

## Implementing Majority Vote Labelling Function

In [26]:
def assign_cluster_labels(y_true, y_pred, k):
    """ Assigns the labels to cluster based on majority voting. """

    cluster_to_label = {}

    for i in range(k):

        # Getting the true labels of the point in this particular cluster
        mask = (y_pred == i)

        if np.sum(mask) >  0:

            # Get the true values of the points in this cluster
            cluster_true_labels = y_true[mask]

            # Find majority label using mode
            unique, counts = np.unique(cluster_true_labels, return_counts=True)
            majority_label = unique[np.argmax(counts)]

            cluster_to_label[i] = majority_label
        else:
            cluster_to_label[i] = -1

    return cluster_to_label 

## Calculating the Accuracy

In [27]:
def calculate_accuracy(y_true, y_pred, cluster_to_label):
    """ Calculate the accuracy based on majority vote labels. """

    y_pred_labels = np.array([cluster_to_label[cluster] for cluster in y_pred])

    correct = np.sum(y_pred_labels == y_true)
    accuracy = (correct / len(y_true)) * 100

    return accuracy

In [28]:
# Calculating Accuracy for Euclidean Distance Metric

euc_labels_norm, euc_centroids_norm, euc_sse_norm, euc_iter_norm, euc_sse_hist_norm = kmeans(X_normalized, K, euclidean_distance, max_iterations=100, random_seed=42)

euclidean_cluster_labels_norm = assign_cluster_labels(y, euc_labels_norm, K)
euclidean_accuracy_norm = calculate_accuracy(y, euc_labels_norm, euclidean_cluster_labels_norm)

print("Cluster to Label Mapping (Euclidean):", euclidean_cluster_labels_norm)
print(f"Euclidean Distance Metric Accuracy: {euclidean_accuracy_norm:.2f}%")

Converged at iteration 80
Cluster to Label Mapping (Euclidean): {0: np.int64(1), 1: np.int64(0), 2: np.int64(3), 3: np.int64(6), 4: np.int64(7), 5: np.int64(4), 6: np.int64(2), 7: np.int64(8), 8: np.int64(0), 9: np.int64(7)}
Euclidean Distance Metric Accuracy: 60.44%


In [29]:
# Calculating Accuracy for Cosine Similarity Distance Metric

cos_labels_norm, cos_centroids_norm, cos_sse_norm, cos_iter_norm, cos_sse_hist_norm = kmeans(X_normalized, K, cosine_distance, max_iterations=100, random_seed=42)

cosine_cluster_labels_norm = assign_cluster_labels(y, cos_labels_norm, K)
cosine_accuracy_norm = calculate_accuracy(y, cos_labels_norm, cosine_cluster_labels_norm)

print("Cluster to Label Mapping (Cosine):", cosine_cluster_labels_norm)
print(f"Cosine Similarity Distance Metric Accuracy: {cosine_accuracy_norm:.2f}%")

Converged at iteration 40
Cluster to Label Mapping (Cosine): {0: np.int64(1), 1: np.int64(0), 2: np.int64(3), 3: np.int64(6), 4: np.int64(1), 5: np.int64(4), 6: np.int64(2), 7: np.int64(8), 8: np.int64(5), 9: np.int64(7)}
Cosine Similarity Distance Metric Accuracy: 62.43%


In [30]:
# Calculating Accuracy for Jaccard Distance Metric

jac_labels_norm, jac_centroids_norm, jac_sse_norm, jac_iter_norm, jac_sse_hist_norm = kmeans(X_normalized, K, jaccard_distance, max_iterations=100, random_seed=42)

jaccard_cluster_labels_norm = assign_cluster_labels(y, jac_labels_norm, K)
jaccard_accuracy_norm = calculate_accuracy(y, jac_labels_norm, jaccard_cluster_labels_norm)

print("Cluster to Label Mapping (Jaccard):", jaccard_cluster_labels_norm)
print(f"Jaccard Distance Metric Accuracy: {jaccard_accuracy_norm:.2f}%")

Converged at iteration 4
Cluster to Label Mapping (Jaccard): {0: np.int64(1), 1: np.int64(0), 2: np.int64(3), 3: np.int64(2), 4: np.int64(8), 5: np.int64(9), 6: np.int64(3), 7: np.int64(5), 8: np.int64(6), 9: np.int64(7)}
Jaccard Distance Metric Accuracy: 50.39%


In [31]:
print("\n1. Running Euclidean K-means (max_iter=50)...")
start_time = time.time()
euc_labels_q3, euc_centroids_q3, euc_sse_q3, euc_iter_q3, euc_sse_hist_q3 = kmeans(
    X_normalized, K, euclidean_distance, max_iterations=50, random_seed=42
)
euc_time_q3 = time.time() - start_time
print(f"   Iterations: {euc_iter_q3}")
print(f"   Time: {euc_time_q3:.2f} seconds")
print(f"   Final SSE: {euc_sse_q3:.2f}")


1. Running Euclidean K-means (max_iter=50)...
Stopped after reaching maximum iterations: 50
   Iterations: 50
   Time: 19.74 seconds
   Final SSE: 391196.60


In [32]:
print("\n2. Running Cosine K-means (max_iter=50)...")
start_time = time.time()
cos_labels_q3, cos_centroids_q3, cos_sse_q3, cos_iter_q3, cos_sse_hist_q3 = kmeans(
    X_normalized, K, cosine_distance, max_iterations=50, random_seed=42
)
cos_time_q3 = time.time() - start_time
print(f"   Iterations: {cos_iter_q3}")
print(f"   Time: {cos_time_q3:.2f} seconds")
print(f"   Final SSE: {cos_sse_q3:.2f}")


2. Running Cosine K-means (max_iter=50)...
Converged at iteration 40
   Iterations: 41
   Time: 19.68 seconds
   Final SSE: 2462.85


In [33]:
print("\n3. Running Jaccard K-means (max_iter=50)...")
start_time = time.time()
jac_labels_q3, jac_centroids_q3, jac_sse_q3, jac_iter_q3, jac_sse_hist_q3 = kmeans(
    X_normalized, K, jaccard_distance, max_iterations=50, random_seed=42
)
jac_time_q3 = time.time() - start_time
print(f"   Iterations: {jac_iter_q3}")
print(f"   Time: {jac_time_q3:.2f} seconds")
print(f"   Final SSE: {jac_sse_q3:.2f}")


3. Running Jaccard K-means (max_iter=50)...
Converged at iteration 4
   Iterations: 5
   Time: 36.89 seconds
   Final SSE: 5508.76


In [34]:
# Creating a comparison table
print("\nConvergence Results:")
print(f"{'Method':<20} {'Iterations':<15} {'Time (seconds)':<20}")
print("-" * 55)
print(f"{'Euclidean':<20} {euc_iter_q3:<15} {euc_time_q3:<20.2f}")
print(f"{'Cosine':<20} {cos_iter_q3:<15} {cos_time_q3:<20.2f}")
print(f"{'Jaccard':<20} {jac_iter_q3:<15} {jac_time_q3:<20.2f}")

# Finding which method requires most iterations
methods_iter = [
    ("Euclidean", euc_iter_q3),
    ("Cosine", cos_iter_q3),
    ("Jaccard", jac_iter_q3)
]
most_iter = max(methods_iter, key=lambda x: x[1])

# Finding which method requires most time
methods_time = [
    ("Euclidean", euc_time_q3),
    ("Cosine", cos_time_q3),
    ("Jaccard", jac_time_q3)
]
most_time = max(methods_time, key=lambda x: x[1])


Convergence Results:
Method               Iterations      Time (seconds)      
-------------------------------------------------------
Euclidean            50              19.74               
Cosine               41              19.68               
Jaccard              5               36.89               


In [35]:
print(f"\nMost iterations required: {most_iter[0]} ({most_iter[1]} iterations)")
print(f"Most time required: {most_time[0]} ({most_time[1]:.2f} seconds)")


Most iterations required: Euclidean (50 iterations)
Most time required: Jaccard (36.89 seconds)


In [36]:
print(f"{most_iter[0]} requires the most iterations to converge.")
print(f"{most_time[0]} requires the most time to converge.")

Euclidean requires the most iterations to converge.
Jaccard requires the most time to converge.


In [37]:
def kmeans_no_centroid_change(X, k, distance_function, max_iterations=100, random_seed=42):
    """K-means that ONLY stops when centroids don't change"""
    centroids = initialize_centroids(X, k, random_seed)
    sse_history = []

    for iteration in range(max_iterations):
        labels = assign_clusters(X, centroids, distance_function)
        current_sse = calculate_sse(X, labels, centroids, distance_function)
        sse_history.append(current_sse)

        # distance-specific centroid update 
        if distance_function == jaccard_distance:
            new_centroids = update_centroids_jaccard(X, labels, k)
        else:
            new_centroids = update_centroids(X, labels, k)

            # normalize centroids for cosine distance
            if distance_function == cosine_distance:
                for j in range(k):
                    norm = np.linalg.norm(new_centroids[j])
                    if norm > 0:
                        new_centroids[j] = new_centroids[j] / norm

        # ONLY check centroid change
        if np.allclose(centroids, new_centroids):
            print(f" Converged (no centroid change) at iteration {iteration + 1}")
            return labels, centroids, current_sse, iteration + 1, sse_history

        centroids = new_centroids

    print(f" Reached max iterations: {max_iterations}")
    return labels, centroids, current_sse, max_iterations, sse_history


In [38]:
def kmeans_sse_increase(X, k, distance_function, max_iterations=100, random_seed=42):
    """K-means that ONLY stops when SSE increases"""
    centroids = initialize_centroids(X, k, random_seed)
    sse_history = []
    prev_sse = float('inf')

    for iteration in range(max_iterations):
        labels = assign_clusters(X, centroids, distance_function)
        current_sse = calculate_sse(X, labels, centroids, distance_function)
        sse_history.append(current_sse)

        # ONLY check SSE increase
        if current_sse > prev_sse:
            print(f" Stopped (SSE increased) at iteration {iteration}")
            return labels, centroids, prev_sse, iteration, sse_history

        # --- distance-specific centroid update ---
        if distance_function == jaccard_distance:
            new_centroids = update_centroids_jaccard(X, labels, k)
        else:
            new_centroids = update_centroids(X, labels, k)

            if distance_function == cosine_distance:
                for j in range(k):
                    norm = np.linalg.norm(new_centroids[j])
                    if norm > 0:
                        new_centroids[j] = new_centroids[j] / norm
        # ------------------------------------------

        centroids = new_centroids
        prev_sse = current_sse

    print(f" Reached max iterations: {max_iterations}")
    return labels, centroids, current_sse, max_iterations, sse_history


In [39]:
def kmeans_max_iter_only(X, k, distance_function, max_iterations=100, random_seed=42):
    """K-means that ONLY stops at max iterations"""
    centroids = initialize_centroids(X, k, random_seed)
    sse_history = []

    for iteration in range(max_iterations):
        labels = assign_clusters(X, centroids, distance_function)
        current_sse = calculate_sse(X, labels, centroids, distance_function)
        sse_history.append(current_sse)

        # --- distance-specific centroid update ---
        if distance_function == jaccard_distance:
            new_centroids = update_centroids_jaccard(X, labels, k)
        else:
            new_centroids = update_centroids(X, labels, k)

            if distance_function == cosine_distance:
                for j in range(k):
                    norm = np.linalg.norm(new_centroids[j])
                    if norm > 0:
                        new_centroids[j] = new_centroids[j] / norm
        # ------------------------------------------

        centroids = new_centroids

    print(f" Reached max iterations: {max_iterations}")
    return labels, centroids, current_sse, max_iterations, sse_history


In [40]:
results_q4 = {}

In [41]:
print("\n1. Euclidean (no centroid change):")
euc_labels_c1, euc_cent_c1, euc_sse_c1, euc_iter_c1, _ = kmeans_no_centroid_change(
    X_normalized, K, euclidean_distance, max_iterations=100, random_seed=42
)
print(f"   SSE: {euc_sse_c1:.2f}, Iterations: {euc_iter_c1}")


1. Euclidean (no centroid change):
 Converged (no centroid change) at iteration 81
   SSE: 391129.76, Iterations: 81


In [42]:
print("\n2. Cosine (no centroid change):")
cos_labels_c1, cos_cent_c1, cos_sse_c1, cos_iter_c1, _ = kmeans_no_centroid_change(
    X_normalized, K, cosine_distance, max_iterations=100, random_seed=42
)
print(f"   SSE: {cos_sse_c1:.2f}, Iterations: {cos_iter_c1}")


2. Cosine (no centroid change):
 Converged (no centroid change) at iteration 41
   SSE: 2462.85, Iterations: 41


In [43]:
print("\n3. Jaccard (no centroid change):")
jac_labels_c1, jac_cent_c1, jac_sse_c1, jac_iter_c1, _ = kmeans_no_centroid_change(
    X_normalized, K, jaccard_distance, max_iterations=100, random_seed=42
)
print(f"   SSE: {jac_sse_c1:.2f}, Iterations: {jac_iter_c1}")


3. Jaccard (no centroid change):
 Converged (no centroid change) at iteration 5
   SSE: 5508.76, Iterations: 5


In [44]:
print("\n1. Euclidean (SSE increase):")
euc_labels_c2, euc_cent_c2, euc_sse_c2, euc_iter_c2, _ = kmeans_sse_increase(
    X_normalized, K, euclidean_distance, max_iterations=100, random_seed=42)

print(f"   SSE: {euc_sse_c2:.2f}, Iterations: {euc_iter_c2}")

print("\n2. Cosine (SSE increase):")
cos_labels_c2, cos_cent_c2, cos_sse_c2, cos_iter_c2, _ = kmeans_sse_increase(
    X_normalized, K, cosine_distance, max_iterations=100, random_seed=42)
print(f"   SSE: {cos_sse_c2:.2f}, Iterations: {cos_iter_c2}")

print("\n3. Jaccard (SSE increase):")
jac_labels_c2, jac_cent_c2, jac_sse_c2, jac_iter_c2, _ = kmeans_sse_increase(
    X_normalized, K, jaccard_distance, max_iterations=50, random_seed=42)

print(f"   SSE: {jac_sse_c2:.2f}, Iterations: {jac_iter_c2}")


1. Euclidean (SSE increase):
 Reached max iterations: 100
   SSE: 391129.76, Iterations: 100

2. Cosine (SSE increase):
 Reached max iterations: 100
   SSE: 2462.85, Iterations: 100

3. Jaccard (SSE increase):
 Reached max iterations: 50
   SSE: 5508.76, Iterations: 50


In [46]:
print("\n1. Euclidean (max iterations):")
euc_labels_c3, euc_cent_c3, euc_sse_c3, euc_iter_c3, _ = kmeans_max_iter_only(
    X_normalized, K, euclidean_distance, max_iterations=50, random_seed=42)

print(f"   SSE: {euc_sse_c3:.2f}, Iterations: {euc_iter_c3}")

print("\n2. Cosine (max iterations):")
cos_labels_c3, cos_cent_c3, cos_sse_c3, cos_iter_c3, _ = kmeans_max_iter_only(
    X_normalized, K, cosine_distance, max_iterations=50, random_seed=42)

print(f"   SSE: {cos_sse_c3:.2f}, Iterations: {cos_iter_c3}")

print("\n3. Jaccard (max iterations):")
jac_labels_c3, jac_cent_c3, jac_sse_c3, jac_iter_c3, _ = kmeans_max_iter_only(
    X_normalized, K, jaccard_distance, max_iterations=50, random_seed=42)

print(f"   SSE: {jac_sse_c3:.2f}, Iterations: {jac_iter_c3}")


1. Euclidean (max iterations):
 Reached max iterations: 50
   SSE: 391196.60, Iterations: 50

2. Cosine (max iterations):
 Reached max iterations: 50
   SSE: 2462.85, Iterations: 50

3. Jaccard (max iterations):
 Reached max iterations: 50
   SSE: 5508.76, Iterations: 50


In [47]:
print(f"{'Distance Metric':<20} {'Condition 1':<20} {'Condition 2':<20} {'Condition 3':<20}")
print(f"{'                ':<20} {'(No Centroid':<20} {'(SSE Increase)':<20} {'(Max Iter=100)':<20}")
print(f"{'                ':<20} {'Change)':<20} {'            ':<20} {'             ':<20}")
print("=" * 80)

print(f"{'Euclidean':<20} {euc_sse_c1:<20.2f} {euc_sse_c2:<20.2f} {euc_sse_c3:<20.2f}")
print(f"{'  Iterations:':<20} {euc_iter_c1:<20} {euc_iter_c2:<20} {euc_iter_c3:<20}")
print()

print(f"{'Cosine':<20} {cos_sse_c1:<20.2f} {cos_sse_c2:<20.2f} {cos_sse_c3:<20.2f}")
print(f"{'  Iterations:':<20} {cos_iter_c1:<20} {cos_iter_c2:<20} {cos_iter_c3:<20}")
print()

print(f"{'Jaccard':<20} {jac_sse_c1:<20.2f} {jac_sse_c2:<20.2f} {jac_sse_c3:<20.2f}")
print(f"{'  Iterations:':<20} {jac_iter_c1:<20} {jac_iter_c2:<20} {jac_iter_c3:<20}")

Distance Metric      Condition 1          Condition 2          Condition 3         
                     (No Centroid         (SSE Increase)       (Max Iter=100)      
                     Change)                                                       
Euclidean            391129.76            391129.76            391196.60           
  Iterations:        81                   100                  50                  

Cosine               2462.85              2462.85              2462.85             
  Iterations:        41                   100                  50                  

Jaccard              5508.76              5508.76              5508.76             
  Iterations:        5                    50                   50                  


In [48]:
print("\nKey Observations:")
print(f"1. Condition 1 (No Centroid Change):")
print(f"   - Euclidean: SSE={euc_sse_c1:.2f}, Iterations={euc_iter_c1}")
print(f"   - Cosine: SSE={cos_sse_c1:.2f}, Iterations={cos_iter_c1}")
print(f"   - Jaccard: SSE={jac_sse_c1:.2f}, Iterations={jac_iter_c1}")

print(f"\n2. Condition 2 (SSE Increase):")
print(f"   - Euclidean: SSE={euc_sse_c2:.2f}, Iterations={euc_iter_c2}")
print(f"   - Cosine: SSE={cos_sse_c2:.2f}, Iterations={cos_iter_c2}")
print(f"   - Jaccard: SSE={jac_sse_c2:.2f}, Iterations={jac_iter_c2}")

print(f"\n3. Condition 3 (Max Iterations=100):")
print(f"   - Euclidean: SSE={euc_sse_c3:.2f}, Iterations={euc_iter_c3}")
print(f"   - Cosine: SSE={cos_sse_c3:.2f}, Iterations={cos_iter_c3}")
print(f"   - Jaccard: SSE={jac_sse_c3:.2f}, Iterations={jac_iter_c3}")



Key Observations:
1. Condition 1 (No Centroid Change):
   - Euclidean: SSE=391129.76, Iterations=81
   - Cosine: SSE=2462.85, Iterations=41
   - Jaccard: SSE=5508.76, Iterations=5

2. Condition 2 (SSE Increase):
   - Euclidean: SSE=391129.76, Iterations=100
   - Cosine: SSE=2462.85, Iterations=100
   - Jaccard: SSE=5508.76, Iterations=50

3. Condition 3 (Max Iterations=100):
   - Euclidean: SSE=391196.60, Iterations=50
   - Cosine: SSE=2462.85, Iterations=50
   - Jaccard: SSE=5508.76, Iterations=50
